# New Zealand Dataset — 分层抽样 (Parquet 优化版)

**目标：** 从 60.7B tokens 中抽取 **1B tokens**，保证每个 CC-MAIN 批次目录都被抽到

**数据格式（已确认）：**
- 每个 `CC-MAIN-YYYY-WW/` 下有 `chunk_*.parquet` 文件
- 字段：`text, id, dump, url, date, file_path, language, language_score, token_count, index`
- **`token_count` 已预计算**，跳过 tiktoken，速度大幅提升

| 项目 | 数值 |
|---|---|
| 总 Token 数 | 60,714,120,289 |
| 总文件数 | 27,168 |
| 目标 Token 数 | 1,000,000,000 |
| 基础抽取比例 | ~1.647% |

In [1]:
!pip install pyarrow tqdm ipywidgets
# cuDF / RAPIDS 通常不要直接 pip install cudf。
# 如果服务器已有 RAPIDS 环境，本 notebook 会自动使用 cuDF；否则自动回退到 PyArrow。

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 52.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 208.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 231.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [ipywidgets]5 [ipywidgets]


In [2]:
# ── Cell 1: 导入 & 配置 ─────────────────────────────────────────
import os, json, random, time
from pathlib import Path
from collections import defaultdict
import pyarrow.parquet as pq
from tqdm.notebook import tqdm

# cuDF 开关：有 GPU/RAPIDS 时使用 cuDF；没有时自动回退 PyArrow
USE_CUDF = True
try:
    import cudf
    CUDF_AVAILABLE = True
    print('cuDF available: True')
except Exception as e:
    cudf = None
    CUDF_AVAILABLE = False
    print(f'cuDF available: False; fallback to PyArrow. Reason: {repr(e)}')

# ════════════════════════════════════════════════════════════════
DATA_ROOT  = Path('/home/jovyan/data/Canada')
OUTPUT_DIR = Path('./nz_sample')
# ════════════════════════════════════════════════════════════════

OUTPUT_FILE = OUTPUT_DIR / 'nz_1B_sample.parquet'
STATS_FILE  = OUTPUT_DIR / 'sampling_stats.json'

TARGET_TOKENS = 1_000_000_000
TOTAL_TOKENS  = 60_714_120_289
SAMPLE_RATIO  = TARGET_TOKENS / TOTAL_TOKENS   # ≈ 1.647%
RANDOM_SEED   = 42

random.seed(RANDOM_SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'DATA_ROOT    : {DATA_ROOT}')
print(f'OUTPUT_FILE  : {OUTPUT_FILE}')
print(f'SAMPLE_RATIO : {SAMPLE_RATIO:.4%}')
print(f'READ_ENGINE  : {"cuDF" if (USE_CUDF and CUDF_AVAILABLE) else "PyArrow"}')

cuDF available: False; fallback to PyArrow. Reason: ModuleNotFoundError("No module named 'cudf'")
DATA_ROOT    : /home/jovyan/data/Canada
OUTPUT_FILE  : nz_sample/nz_1B_sample.parquet
SAMPLE_RATIO : 1.6471%
READ_ENGINE  : PyArrow


In [3]:
# ── Cell 2: 扫描所有 CC-MAIN 目录，找出 .parquet 文件 ──────────
dir_files = {}   # { 'CC-MAIN-2013-20': [Path, Path, ...] }

for batch_dir in sorted(DATA_ROOT.iterdir()):
    if not batch_dir.is_dir() or not batch_dir.name.startswith('CC-MAIN'):
        continue
    files = sorted(batch_dir.glob('*.parquet'))
    if files:
        dir_files[batch_dir.name] = files

all_dirs = sorted(dir_files.keys())
n_dirs   = len(all_dirs)
total_files_scanned = sum(len(v) for v in dir_files.values())

print(f'发现 CC-MAIN 目录数 : {n_dirs}')
print(f'扫描到 parquet 文件 : {total_files_scanned}')
print(f'\n各目录文件数（前 10 条）:')
for d in all_dirs[:10]:
    print(f'  {d}: {len(dir_files[d])} 个文件')
if n_dirs > 10:
    print(f'  ... 还有 {n_dirs-10} 个目录')

发现 CC-MAIN 目录数 : 108
扫描到 parquet 文件 : 336

各目录文件数（前 10 条）:
  CC-MAIN-2013-20: 2 个文件
  CC-MAIN-2013-48: 2 个文件
  CC-MAIN-2014-10: 2 个文件
  CC-MAIN-2014-15: 2 个文件
  CC-MAIN-2014-23: 2 个文件
  CC-MAIN-2014-35: 2 个文件
  CC-MAIN-2014-41: 2 个文件
  CC-MAIN-2014-42: 2 个文件
  CC-MAIN-2014-49: 2 个文件
  CC-MAIN-2014-52: 2 个文件
  ... 还有 98 个目录


In [4]:
# ── Cell 3: 利用 progress.json 获取每个目录的真实 token 数 ─────
# 比按文件数比例更准确：直接按 token 占比分配配额

dir_tokens = {}
for d in all_dirs:
    progress = DATA_ROOT / d / 'progress.json'
    if progress.exists():
        with open(progress) as f:
            info = json.load(f)
        dir_tokens[d] = info.get('total_tokens', 0)
    else:
        dir_tokens[d] = 0   # 没 progress.json 的稍后用文件数估算

sum_known_tokens = sum(dir_tokens.values())
print(f'从 progress.json 收集到的总 tokens: {sum_known_tokens:,}')
print(f'元数据声明的总 tokens          : {TOTAL_TOKENS:,}')
print(f'\nToken 数最大的 5 个目录:')
for d, t in sorted(dir_tokens.items(), key=lambda x: -x[1])[:5]:
    print(f'  {d}: {t:>14,} tokens')

从 progress.json 收集到的总 tokens: 209,731,614,505
元数据声明的总 tokens          : 60,714,120,289

Token 数最大的 5 个目录:
  CC-MAIN-2023-40:  3,626,422,933 tokens
  CC-MAIN-2023-23:  3,294,041,908 tokens
  CC-MAIN-2022-49:  3,200,695,907 tokens
  CC-MAIN-2023-06:  3,150,666,603 tokens
  CC-MAIN-2023-14:  3,115,109,450 tokens


In [5]:
# ── Cell 4: 分层配额计算（按 token 占比，每目录至少 1 个文件）─
# 每目录目标 tokens = SAMPLE_RATIO × 该目录总 tokens
# 抽样文件数 = ceil(目录文件数 × SAMPLE_RATIO)，至少 1

import math

quota_per_dir = {}
for d in all_dirs:
    n_files = len(dir_files[d])
    # 按文件比例抽，至少 1 个，不超过实际文件数
    n_to_pick = max(1, min(math.ceil(n_files * SAMPLE_RATIO), n_files))
    quota_per_dir[d] = n_to_pick

total_selected_files = sum(quota_per_dir.values())
print(f'实际选取文件数  : {total_selected_files}')
print(f'\n各目录配额（前 10 条）:')
for d in all_dirs[:10]:
    print(f'  {d}: {len(dir_files[d])} 文件 → 抽 {quota_per_dir[d]} 个')

实际选取文件数  : 108

各目录配额（前 10 条）:
  CC-MAIN-2013-20: 2 文件 → 抽 1 个
  CC-MAIN-2013-48: 2 文件 → 抽 1 个
  CC-MAIN-2014-10: 2 文件 → 抽 1 个
  CC-MAIN-2014-15: 2 文件 → 抽 1 个
  CC-MAIN-2014-23: 2 文件 → 抽 1 个
  CC-MAIN-2014-35: 2 文件 → 抽 1 个
  CC-MAIN-2014-41: 2 文件 → 抽 1 个
  CC-MAIN-2014-42: 2 文件 → 抽 1 个
  CC-MAIN-2014-49: 2 文件 → 抽 1 个
  CC-MAIN-2014-52: 2 文件 → 抽 1 个


In [6]:
# ── Cell 5: 随机选取具体文件 ───────────────────────────────────
selected_files = []
for d in all_dirs:
    pool   = dir_files[d]
    quota  = quota_per_dir[d]
    chosen = random.sample(pool, quota)
    selected_files.extend(chosen)

random.shuffle(selected_files)

print(f'最终选取文件数 : {len(selected_files)}')
print(f'\n样本（前 5 个文件）:')
for f in selected_files[:5]:
    print(f'  {f.parent.name}/{f.name}')

最终选取文件数 : 108

样本（前 5 个文件）:
  CC-MAIN-2018-22/chunk_2.parquet
  CC-MAIN-2024-18/chunk_3.parquet
  CC-MAIN-2014-52/chunk_0.parquet
  CC-MAIN-2018-09/chunk_2.parquet
  CC-MAIN-2021-39/chunk_1.parquet


In [7]:
# ── Cell 6: 主抽样循环（cuDF 优先；PyArrow fallback；流式写入）──────
import pyarrow as pa
from collections import defaultdict
from tqdm import tqdm

READ_WITH_CUDF = bool(USE_CUDF and CUDF_AVAILABLE)
print(f"读取引擎: {'cuDF / GPU' if READ_WITH_CUDF else 'PyArrow / CPU'}")

def read_token_counts(fp):
    """只读取 token_count 列，用于快速决定需要截取多少行。"""
    if READ_WITH_CUDF:
        gdf = cudf.read_parquet(str(fp), columns=['token_count'])
        # cumsum 很小，直接转到 CPU 做 argmax，避免后续逻辑复杂化
        return gdf['token_count'].to_numpy()
    table = pq.read_table(fp, columns=['token_count'])
    return table.column('token_count').to_numpy()


def read_full_slice_as_arrow(fp, take_n):
    """读取完整数据，并只保留前 take_n 行；统一返回 PyArrow Table 给 ParquetWriter。"""
    if READ_WITH_CUDF:
        gdf = cudf.read_parquet(str(fp))
        if take_n < len(gdf):
            gdf = gdf.iloc[:take_n]
        table = gdf.to_arrow()
        del gdf
        return table

    table = pq.read_table(fp)
    if take_n < table.num_rows:
        table = table.slice(0, take_n)
    return table


tokens_per_dir_quota = TARGET_TOKENS // n_dirs
print(f"每个目录目标贡献: {tokens_per_dir_quota:,} tokens")

files_by_dir = defaultdict(list)
for fp in selected_files:
    files_by_dir[fp.parent.name].append(fp)

total_tokens_written = 0
total_rows_written   = 0
files_done           = 0
t0                   = time.time()
stats_per_dir        = defaultdict(lambda: {'files': 0, 'tokens': 0, 'rows': 0})

pbar = tqdm(total=TARGET_TOKENS, unit='tok', unit_scale=True, desc='抽样进度', mininterval=0.5)

writer = None   # 流式 ParquetWriter

for dir_name in all_dirs:
    dir_quota = tokens_per_dir_quota
    dir_collected = 0

    for fp in files_by_dir[dir_name]:
        if dir_collected >= dir_quota:
            break
        try:
            token_counts = read_token_counts(fp)
        except Exception as e:
            print(f'[WARN] 读取 token_count 失败，跳过 {fp.name}: {e}')
            continue

        if len(token_counts) == 0:
            continue

        cumsum = token_counts.cumsum()
        need = dir_quota - dir_collected

        if cumsum[-1] <= need:
            take_n = len(token_counts)
            take_tokens = int(cumsum[-1])
        else:
            idx = int((cumsum >= need).argmax()) + 1
            take_n = idx
            take_tokens = int(cumsum[idx-1])

        try:
            table = read_full_slice_as_arrow(fp, take_n)
        except Exception as e:
            print(f'[WARN] 读取完整数据失败，跳过 {fp.name}: {e}')
            continue

        if writer is None:
            writer = pq.ParquetWriter(OUTPUT_FILE, table.schema, compression='snappy')

        writer.write_table(table)
        del table, token_counts, cumsum

        dir_collected        += take_tokens
        total_tokens_written += take_tokens
        total_rows_written   += take_n
        stats_per_dir[dir_name]['files']  += 1
        stats_per_dir[dir_name]['tokens'] += take_tokens
        stats_per_dir[dir_name]['rows']   += take_n
        files_done += 1
        pbar.update(take_tokens)

        if files_done % 5 == 0:
            elapsed = time.time() - t0
            tps = total_tokens_written / max(elapsed, 1e-6)
            print(f"  [{files_done}] {dir_name}: 累计 {total_tokens_written/1e6:.0f}M tokens, 覆盖 {len(stats_per_dir)}/{n_dirs} 目录, 速度 {tps/1e6:.1f}M tok/s")

if writer is not None:
    writer.close()
pbar.close()

elapsed_total = time.time() - t0
print(f'\n{"="*55}')
print(f'✅ 抽样完成！')
print(f'   使用引擎      : {"cuDF / GPU" if READ_WITH_CUDF else "PyArrow / CPU"}')
print(f'   总 tokens     : {total_tokens_written:,}')
print(f'   总行数        : {total_rows_written:,}')
print(f'   处理文件数    : {files_done}')
print(f'   覆盖目录数    : {len(stats_per_dir)} / {n_dirs}')
print(f'   总耗时        : {elapsed_total/60:.1f} 分钟')
print(f'   平均速度      : {total_tokens_written/elapsed_total/1e6:.1f}M tok/s')
print(f'   输出文件      : {OUTPUT_FILE}')
print(f'   输出大小      : {OUTPUT_FILE.stat().st_size/1024/1024:.1f} MB')

读取引擎: PyArrow / CPU
每个目录目标贡献: 9,259,259 tokens


抽样进度:   5%|▍         | 46.3M/1.00G [01:23<29:13, 544ktok/s]

  [5] CC-MAIN-2014-23: 累计 46M tokens, 覆盖 5/108 目录, 速度 0.6M tok/s


抽样进度:   9%|▉         | 92.7M/1.00G [02:41<24:58, 606ktok/s]

  [10] CC-MAIN-2014-52: 累计 93M tokens, 覆盖 10/108 目录, 速度 0.6M tok/s


...

  [100] CC-MAIN-2024-42: 累计 926M tokens, 覆盖 100/108 目录, 速度 0.6M tok/s


抽样进度:  97%|█████████▋| 972M/1.00G [25:57<00:38, 715ktok/s]

  [105] CC-MAIN-2025-13: 累计 972M tokens, 覆盖 105/108 目录, 速度 0.6M tok/s


抽样进度: 1.00Gtok [26:50, 621ktok/s]                         


✅ 抽样完成！
   使用引擎      : PyArrow / CPU
   总 tokens     : 1,000,266,490
   总行数        : 1,463,015
   处理文件数    : 108
   覆盖目录数    : 108 / 108
   总耗时        : 26.8 分钟
   平均速度      : 0.6M tok/s
   输出文件      : nz_sample/nz_1B_sample.parquet
   输出大小      : 2754.0 MB


In [10]:
# ── Cell 7: 保存统计报告 ────────────────────────────────────────
stats = {
    'dataset': 'Canada',
    'sampling_method': 'stratified_by_cc_main_batch_min1_per_dir',
    'random_seed': RANDOM_SEED,
    'target_tokens': TARGET_TOKENS,
    'total_tokens_written': total_tokens_written,
    'total_rows_written': total_rows_written,
    'files_processed': files_done,
    'dirs_covered': len(stats_per_dir),
    'total_dirs': n_dirs,
    'elapsed_minutes': round(elapsed_total / 60, 2),
    'output_size_mb': round(OUTPUT_FILE.stat().st_size / 1024 / 1024, 1),
    'per_directory': {k: v for k, v in sorted(stats_per_dir.items())}
}

with open(STATS_FILE, 'w') as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)

print(f'统计报告已保存: {STATS_FILE}\n')
print(f'{"目录":<25} {"文件":>5} {"Tokens":>14} {"行数":>10}')
print('-' * 60)
for d, s in sorted(stats_per_dir.items()):
    print(f'{d:<25} {s["files"]:>5} {s["tokens"]:>14,} {s["rows"]:>10,}')
print('-' * 60)
print(f'{"合计":<25} {files_done:>5} {total_tokens_written:>14,} {total_rows_written:>10,}')

FileNotFoundError: [Errno 2] No such file or directory: 'nz_sample/nz_1B_sample.parquet'

In [9]:
# ── Cell 8: 验证输出文件 ────────────────────────────────────────
verify = pq.read_table(OUTPUT_FILE)
print(f'输出文件总行数 : {verify.num_rows:,}')
print(f'输出文件列名   : {verify.column_names}')
print(f'token_count 总和: {sum(verify.column("token_count").to_numpy()):,}')

print('\n前 3 条记录预览：')
for i in range(3):
    row = verify.slice(i, 1).to_pylist()[0]
    print(f'\n── 记录 {i+1} ──')
    print(f'  dump  : {row["dump"]}')
    print(f'  url   : {row["url"]}')
    print(f'  tokens: {row["token_count"]}')
    print(f'  text  : {row["text"][:150]}...')

输出文件总行数 : 1,463,015
输出文件列名   : ['text', 'id', 'dump', 'url', 'date', 'file_path', 'language', 'language_score', 'token_count', 'index']
token_count 总和: 1,000,266,490

前 3 条记录预览：

── 记录 1 ──
  dump  : CC-MAIN-2013-20
  url   : http://aangirfan.blogspot.ca/2008/12/gambia-and-organised-crime.html
  tokens: 1848
  text  : According to reports, Mr Jammeh recently told supporters he would “cut off the head” of any homosexual caught in the Gambia.
1. The Gambia is ruled by...

── 记录 2 ──
  dump  : CC-MAIN-2013-20
  url   : http://alexovetjkin.blogspot.ca/2008/11/scouting-report-on-capitals.html
  tokens: 745
  text  : Sutter's son, Brandon, returned to the Hurricanes' lineup Wednesday night for the first time since he suffered a concussion against the Islanders on O...

── 记录 3 ──
  dump  : CC-MAIN-2013-20
  url   : http://anysportanytime.ca/tag/shannon-kleibrink/
  tokens: 335
  text  : This is one of the few weekends every year that I get up in the middle of the night to watch sports, but h

## 速度预估

| 阶段 | 估算 |
|---|---|
| Cell 2-5 扫描+配额+选文件 | < 5 秒 |
| **Cell 6 主抽样** | **1-3 分钟** |
| Cell 7-8 输出统计 + 验证 | < 5 秒 |
| **总计** | **≈ 1.5-3.5 分钟** |

**为什么这么快：**
- Parquet 列存，直接读 `token_count` 列做累加，**完全跳过 tiktoken**
- 整个文件一次读入（pyarrow 用 C++ 实现）
- 命中目标后立即停止，可能只处理 ~100 个文件

## 输出文件
- `nz_1B_sample.parquet` — 样本（Snappy 压缩，~3-5 GB）
- `sampling_stats.json` — 每个 CC-MAIN 批次的抽取统计

## 论文里的描述
> *We employ stratified random sampling across Common Crawl batches (CC-MAIN-YYYY-WW), allocating at least one file per batch to ensure temporal coverage. The number of files sampled per batch is proportional to the batch's file count. Token counts are taken directly from the pre-computed `token_count` field. Random seed = 42 for reproducibility.*